# subagent

> Delegating a question whose *answer* is small and whose *working* is enormous.

The pattern is aai-coding's `looker`, generalised. `looker` answers a question about a
screenshot by sending it to a fresh ephemeral thread with a narrow charter and returning
only the answer, so the images never enter the caller's context. The images are incidental;
the shape is the idea.

"Find every place we already handle this" is the canonical case. It costs twenty tool
results and yields three lines of conclusion, and those twenty results then sit in the
context window for the rest of the session, getting summarised at compaction time and
paid for on every subsequent turn. Handing it to a sub-agent means the search happens, the
conclusion comes back, and the twenty results are thrown away with the conversation that
produced them.

Two design choices worth stating:

**On the same engine.** `Backend.spawn` shares the loaded model, so delegation never means
a second multi-gigabyte load or a second provider client.

**On the cheap model, by default.** Routing already sends `subagent` to the local model.
Fan-out is a search problem, not a reasoning problem: reading twenty results and reporting
which three matter is comfortably within a small model, and it is the exact work the
routing policy exists to keep off the wire.

The sub-agent gets **no write tools**. Not a policy that can be relaxed by argument: a
delegated question is a question, and an agent nobody is watching should not be editing
files. If work needs doing, the answer comes back and the main agent does it under the
approval the user is actually looking at.


In [ ]:
#| default_exp subagent

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from ramabana.core import agent_err
from ramabana.tools import WRITE_TOOLS, clip, MAX_TOOL_CHARS

In [ ]:
#| export
SUB_MAX_STEPS = 12

In [ ]:
#| export
SUB_SP = """You are a research sub-agent inside a Python IDE. Another agent has delegated one \
question to you because answering it takes many tool calls and the answer is short.

- Answer exactly the question asked. Nothing else.
- Use your tools as much as you need; nobody is paying attention to how many calls it takes.
- Report what you found, with file paths and line numbers, not what you infer or expect.
- If the answer is that there is nothing, say so plainly. A confident wrong answer is far \
worse than "no matches, and here is what I searched for".
- You cannot edit anything. If the answer implies a change, describe the change and stop.
- `inspect_python` answers questions about the user's live variables without changing them. \
Its default scope is a sandbox that refuses most library calls; pass `scope='overlay'` to \
get the real interpreter. Use it rather than guessing at what is in memory."""

In [ ]:
#| export
# A sub-agent does not get to spawn sub-agents. Not a safety rule so much as an economic
# one: recursion here is a fan-out tree whose width nobody chose, and the second level
# never has enough context to ask a good question anyway.
NO_SUB = frozenset({'delegate_search', 'delegate_parallel'})

In [ ]:
#| export
def read_only(tools):
    "The tools a sub-agent may have: everything that only reads, and no delegation of its own."
    return [t for t in tools if getattr(t, '__name__', '') not in (WRITE_TOOLS | NO_SUB)]

In [ ]:
#| export
def delegate(backend, question, tools=(), sp=SUB_SP, max_steps=SUB_MAX_STEPS):
    """Ask `question` in a throwaway conversation on `backend`'s engine. Returns the answer text.

    The conversation is closed in a `finally` because the whole benefit is that it does not
    outlive the question -- a sub-agent whose context leaks back into the session is just a
    slower way of doing the work inline.
    """
    sub = None
    try:
        sub = backend.spawn(sp=sp, tools=read_only(tools))
        if hasattr(sub, 'max_steps'): sub.max_steps = max_steps
        return sub.send(question)
    except Exception as e:
        return f'delegation failed: {agent_err(e)}'
    finally:
        if sub is not None:
            try: sub.close()
            except Exception: pass

In [ ]:
#| export
def delegate_many(backend, questions, tools=(), sp=SUB_SP, max_steps=SUB_MAX_STEPS, n_workers=4):
    """Ask several questions at once. Returns answers in the order the questions were given.

    Whether this is genuinely parallel depends on what is underneath, and it is worth being
    exact rather than optimistic:

    - **Generation** overlaps on a cloud backend, where each sub-agent is its own HTTP
      request. On a local engine it does not -- litert holds one conversation at a time --
      so local fan-out is run one after another rather than racing for the same engine and
      finding out what happens.
    - **Tool work** overlaps either way, and is usually the bulk of it: three sub-agents
      each doing six searches is eighteen searches, and they do not wait for each other.
      Under a concurrent kernel (`Host.kernel_kind == 'ipymini'`) that includes
      `inspect_python`, which is the case that used to be hopeless -- an inspection queued
      behind the user's running cell, and then behind the other two sub-agents' inspections.

    The point of the whole thing is context, not speed. Three questions answered in
    parallel cost the caller three short answers instead of sixty tool results.
    """
    qs = list(questions)
    if not qs: return []
    if len(qs) == 1 or getattr(backend.spec, 'local', False) or n_workers < 2:
        return [delegate(backend, q, tools, sp, max_steps) for q in qs]
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(min(n_workers, len(qs))) as ex:
        return list(ex.map(lambda q: delegate(backend, q, tools, sp, max_steps), qs))

In [ ]:
#| export
def subagent_tools(get_backend, get_tools):
    """The `delegate` tool, bound to whatever backend routing says sub-agents run on.

    Both arguments are callables so a model switch mid-session is picked up: the tool the
    model is holding must not be pinned to the backend that happened to be current when
    the tool list was built.
    """

    def delegate_search(question: str) -> str:
        """Hand a broad search question to a sub-agent and get back only its conclusion.

        Use this when answering would take many `search_code` / `view_file` / `read_url` /
        `inspect_python` calls whose results you do not need to keep -- "where else do we
        do X", "which files import Y", "what shape is everything in this namespace". The
        sub-agent has your read-only tools and none of your write tools, and its working is
        discarded, so the cost to your context is one question and one answer.

        Ask one self-contained question. The sub-agent cannot see this conversation.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        return clip(delegate(b, question, get_tools()), MAX_TOOL_CHARS)

    def delegate_parallel(questions: str) -> str:
        """Hand several independent questions to sub-agents at once, and get back every answer.

        `questions` is a JSON array of strings, e.g.
          ["which files import fastllm?", "where is compaction triggered?", "what is df's shape?"]

        Use it when you have two or more questions that do not depend on each other. They
        run concurrently, each in its own throwaway conversation with your read-only tools,
        so three questions cost you three short answers rather than the sixty tool results
        it would take to answer them yourself.

        Every question must be self-contained: a sub-agent cannot see this conversation or
        the other questions.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        try:
            qs = json.loads(questions) if isinstance(questions, str) else list(questions)
            if not isinstance(qs, list) or not all(isinstance(q, str) for q in qs):
                raise ValueError('expected a JSON array of strings')
        except Exception as e:
            return f'could not parse questions: {agent_err(e)}'
        if not qs: return 'no questions given'
        answers = delegate_many(b, qs, get_tools())
        return clip('\n\n'.join(f'### {q}\n{a}' for q, a in zip(qs, answers)), MAX_TOOL_CHARS * 2)

    return [delegate_search, delegate_parallel]

## Tests


In [ ]:
# A delegate that can write is a delegate that can wreck the repo, so `read_only` is the
# gate: every write tool, and delegation itself, is taken away before it is handed over.
from ramabana.testing import MemHost
from ramabana.tools import WRITE_TOOLS, tools_for
full = tools_for(MemHost({'/proj/a.py': 'x = 1\n'}))
sub = [t.__name__ for t in read_only(full)]
print('the agent has :', sorted(t.__name__ for t in full))
print('a sub-agent has:', sorted(sub))
assert not (set(sub) & WRITE_TOOLS), 'a sub-agent must never get a write tool'

In [ ]:
# The delegate tools are bound to *callables*, not to a backend, so switching model
# mid-session is picked up by a tool the model is already holding.
from ramabana.testing import FakeBackend, SPEC
be = FakeBackend(SPEC, replies=['three files import it'])
be.start()
ts = {t.__name__: t for t in subagent_tools(lambda: be, lambda: read_only(full))}
print('delegation tools:', sorted(ts))
print(ts['delegate_search']('which files import fastllm?'))

In [ ]:
# The whole benefit is that the conversation does not outlive the question.
be2 = FakeBackend(SPEC, replies=['the answer'])
be2.start()
print(delegate(be2, 'what shape is df?', read_only(full)))
print('sub-conversations spawned:', len(be2.spawned))